# V14 – Betriebssysteme Teil 2: Praxis (`os`-Modul)

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- mit **`os.listdir(pfad)`** den Inhalt eines Ordners als Liste auslesen,
- mit **`os.path.isfile(pfad)`** und **`os.path.isdir(pfad)`** prüfen, ob ein Eintrag eine Datei oder ein Verzeichnis ist,
- mit **`os.path.getsize(pfad)`** die Größe einer Datei in Bytes ermitteln,
- mit **`os.path.join(basis, name)`** plattformunabhängig Pfade zusammenbauen,
- mit **`os.makedirs(pfad, exist_ok=True)`** einen Ordner anlegen und mit **`shutil.rmtree(pfad)`** rekursiv wieder entfernen.

## ⏱️ Zeitbudget
Ca. 25 Minuten.

## 🧭 TL;DR
Wir legen uns im Notebook-Verzeichnis einen temporären Ordner `temp_messdaten/` mit ein paar Dummy-Messdateien an und üben mit dem `os`-Modul alle Operationen, die du später in den Aufgaben brauchst. Am Ende räumen wir den Ordner wieder auf, damit dein Arbeitsverzeichnis sauber bleibt.

## 📦 Voraussetzungen
- [01_theorie.ipynb](01_theorie.ipynb) — insbesondere Abschnitt 5 (Dateisystem)
- Python-Basics: Listen, Schleifen, f-Strings


## 0. Was ist das `os`-Modul?

Das Modul **`os`** gehört zur Python-Standardbibliothek und liefert den plattformunabhängigen Zugang zu Betriebssystem-Funktionen. Egal ob du unter Windows, Linux oder macOS arbeitest: `os.listdir` funktioniert überall gleich und liefert dir einen Ordner-Inhalt als Liste von Strings.

Der wichtigste Untermodul-Bereich ist **`os.path`**. Er enthält Hilfsfunktionen, die Pfade als Strings analysieren: existiert der Pfad überhaupt? Ist es eine Datei? Ein Verzeichnis? Wie groß ist sie? Wir nutzen in diesem Notebook fünf dieser Funktionen — mehr brauchst du für die Aufgaben nicht.


### Vorbereitung: aktuelles Arbeitsverzeichnis

Bevor wir anfangen, vergewissern wir uns, in welchem Ordner das Notebook gerade läuft. **`os.getcwd()`** („get current working directory") liefert den Pfad zurück. Relative Pfade wie `"temp_messdaten"` werden Python immer relativ zu *diesem* Ordner aufgelöst.


In [1]:
import os

print("Arbeitsverzeichnis:", os.getcwd())


Arbeitsverzeichnis: C:\_data\_Git\github_private\maschinenbau-vorlsung-25_26\lessons\V14-Betriebssysteme-Rechnerarchitektur-Teil2


### Setup: Temp-Ordner mit Dummy-Dateien anlegen

Damit das Notebook auf jedem Rechner läuft — unabhängig davon, was sonst so in deinem Arbeitsverzeichnis herumliegt — erzeugen wir uns einen eigenen **temporären Ordner** mit ein paar Dummy-Dateien. Die folgende Zelle nutzt **`os.makedirs(pfad, exist_ok=True)`**, um den Ordner sicher anzulegen: Existiert er bereits, passiert nichts; existiert er nicht, wird er erzeugt. Anschließend schreiben wir fünf kleine Dateien hinein und einen Unterordner `archiv/`.

> [!NOTE]
> Das Argument `newline="\n"` beim Öffnen sorgt dafür, dass die Dateigrößen auf Windows und Unix identisch sind — sonst würde Windows automatisch `\r\n` einfügen, und die Asserts weiter unten würden brechen.


In [2]:
import os

temp_ordner = "temp_messdaten"
os.makedirs(temp_ordner, exist_ok=True)

# Simulierte Mess-Dateien anlegen
dateien = {
    "messung_01.csv": "Zeit,Drehzahl\n0,1200\n1,1210\n2,1205\n",
    "messung_02.csv": "Zeit,Drehzahl\n0,1190\n1,1202\n",
    "messung_03.csv": "Zeit,Drehzahl\n0,1220\n1,1225\n2,1219\n3,1230\n",
    "kalibrierung.txt": "Offset: 0.02\nSteigung: 1.001\n",
    "logbuch.log": "08:00 gestartet\n08:05 Warnung\n",
}
for name, inhalt in dateien.items():
    # newline="\n" erzwingt Unix-Zeilenenden unabhaengig vom Betriebssystem
    with open(os.path.join(temp_ordner, name), "w", encoding="utf-8", newline="\n") as f:
        f.write(inhalt)

# Zusaetzlich einen Unterordner anlegen, damit wir isdir pruefen koennen
os.makedirs(os.path.join(temp_ordner, "archiv"), exist_ok=True)

print(f"Ordner '{temp_ordner}' vorbereitet mit {len(dateien)} Dateien + 1 Unterordner.")


Ordner 'temp_messdaten' vorbereitet mit 5 Dateien + 1 Unterordner.


## 1. `os.listdir` — den Ordner-Inhalt auflisten

Die Funktion **`os.listdir(pfad)`** liefert dir den Inhalt des angegebenen Ordners als Liste von Strings zurück. Jeder String ist der Name eines Eintrags — *ohne* den Pfad davor und *ohne* Unterscheidung, ob es sich um eine Datei oder einen Unterordner handelt.

Die Reihenfolge der Einträge ist **nicht garantiert**. Auf manchen Systemen erhältst du sie alphabetisch sortiert, auf anderen in der Reihenfolge, in der sie ins Dateisystem aufgenommen wurden. Wenn du deterministische Reihenfolge brauchst, sortiere selbst: `sorted(os.listdir(pfad))`.

> [!WARNING]
> `os.listdir` liefert **nur Namen**, keine vollständigen Pfade. Wenn du später auf diese Einträge zugreifen willst (z. B. mit `os.path.getsize`), musst du den Ordnernamen davor setzen — dafür ist `os.path.join` da.


In [3]:
import os

eintraege = os.listdir("temp_messdaten")

print(f"'temp_messdaten' enthaelt {len(eintraege)} Eintraege:")
for name in sorted(eintraege):
    print(" -", name)


'temp_messdaten' enthaelt 6 Eintraege:
 - archiv
 - kalibrierung.txt
 - logbuch.log
 - messung_01.csv
 - messung_02.csv
 - messung_03.csv


## 2. `os.path.isfile` und `os.path.isdir` — Datei oder Ordner?

Da `listdir` keinen Hinweis darauf liefert, ob ein Eintrag Datei oder Ordner ist, brauchen wir zwei Prüffunktionen. **`os.path.isfile(pfad)`** gibt `True` zurück, wenn der Pfad auf eine existierende Datei zeigt. **`os.path.isdir(pfad)`** gibt `True` zurück, wenn er auf ein existierendes Verzeichnis zeigt. Beide liefern `False`, wenn der Pfad nicht existiert (sie werfen keine Exception).

Wichtig: beide Funktionen brauchen den **vollen Pfad** relativ zu `os.getcwd()`, nicht nur den Namen aus `listdir`. Wir bauen den Pfad also aus Ordnername und Eintragsname zusammen.


In [4]:
import os

basis = "temp_messdaten"

for name in sorted(os.listdir(basis)):
    voller_pfad = os.path.join(basis, name)
    if os.path.isfile(voller_pfad):
        kennzeichen = "DATEI"
    elif os.path.isdir(voller_pfad):
        kennzeichen = "ORDNER"
    else:
        kennzeichen = "???"
    print(f"  {kennzeichen:6s}  {voller_pfad}")


  ORDNER  temp_messdaten\archiv
  DATEI   temp_messdaten\kalibrierung.txt
  DATEI   temp_messdaten\logbuch.log
  DATEI   temp_messdaten\messung_01.csv
  DATEI   temp_messdaten\messung_02.csv
  DATEI   temp_messdaten\messung_03.csv


### 🧩 Fill-in 1: nur die Dateien

Fülle unten den TODO-Platzhalter so, dass die Variable `nur_dateien` **ausschließlich die Dateinamen** (also nicht den Unterordner `archiv`) aus `temp_messdaten` enthält. Nutze `os.listdir`, `os.path.join` und `os.path.isfile`.

**Erwartet:** Eine Liste mit 5 Einträgen, sortiert: `['kalibrierung.txt', 'logbuch.log', 'messung_01.csv', 'messung_02.csv', 'messung_03.csv']`.


In [5]:
import os

basis = "temp_messdaten"

# TODO: Baue eine Liste der Eintrage, die wirklich Dateien sind.
nur_dateien = []

print("Dateien:", sorted(nur_dateien))

# ▶️ Selbstkontrolle
try:
    erwartet = ["kalibrierung.txt", "logbuch.log", "messung_01.csv", "messung_02.csv", "messung_03.csv"]
    assert sorted(nur_dateien) == erwartet, f"Erwartet {erwartet}, bekommen: {sorted(nur_dateien)}"
    print("✅ Richtig!")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Variable `nur_dateien` fehlt noch.")


Dateien: []
❌ Noch nicht ganz: Erwartet ['kalibrierung.txt', 'logbuch.log', 'messung_01.csv', 'messung_02.csv', 'messung_03.csv'], bekommen: []


## 3. `os.path.getsize` — wie groß ist eine Datei?


**`os.path.getsize(pfad)`** liefert die Größe einer Datei in **Byte** zurück. Für Ordner ist der Rückgabewert plattformabhängig und meist wenig aussagekräftig — wir prüfen deswegen mit `os.path.isfile` vorher, bevor wir die Größe abfragen.

Wenn der Pfad nicht existiert, wirft `getsize` einen `FileNotFoundError`. Das ist bewusst so: Eine nicht existierende Datei hat keine Größe, und jeder andere Rückgabewert (z. B. 0) wäre irreführend.


In [6]:
import os

basis = "temp_messdaten"

for name in sorted(os.listdir(basis)):
    voller_pfad = os.path.join(basis, name)
    if os.path.isfile(voller_pfad):
        groesse = os.path.getsize(voller_pfad)
        print(f"  {name:20s} {groesse:4d} Byte")


  kalibrierung.txt       29 Byte
  logbuch.log            30 Byte
  messung_01.csv         35 Byte
  messung_02.csv         28 Byte
  messung_03.csv         42 Byte


### 🧩 Fill-in 2: die größte Datei finden

Ermittle unten den **Namen** der größten Datei im Ordner `temp_messdaten` und weise ihn der Variable `groesste_datei` zu. Die Dateigröße in Byte soll in `groesste_byte` landen.

**Erwartet:** `groesste_datei == "messung_03.csv"` mit `groesste_byte == 42`.


In [7]:
import os

basis = "temp_messdaten"

groesste_datei = ""
groesste_byte = 0

# TODO: iteriere ueber die Eintraege, merke dir Name und Byte-Zahl der groessten Datei

print(f"Groesste Datei: {groesste_datei} ({groesste_byte} Byte)")

# ▶️ Selbstkontrolle
try:
    assert groesste_datei == "messung_03.csv", f"Erwartet 'messung_03.csv', bekommen: '{groesste_datei}'"
    assert groesste_byte == 42, f"Erwartet 42 Byte, bekommen: {groesste_byte}"
    print("✅ Richtig!")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Eine Variable fehlt noch.")


Groesste Datei:  (0 Byte)
❌ Noch nicht ganz: Erwartet 'messung_03.csv', bekommen: ''


## 4. `os.path.join` — Pfade sauber zusammenbauen

Du hast `os.path.join` oben bereits zweimal angewendet. Warum ist diese Funktion so wichtig — könntest du nicht einfach `basis + "/" + name` schreiben? Die Antwort: unter Windows ist der Pfad-Trenner `\` (Backslash), unter Linux und macOS `/`. **`os.path.join`** erkennt die Plattform und setzt automatisch den richtigen Trenner ein. Dein Code läuft damit auf allen Betriebssystemen ohne Anpassung.

Die Funktion nimmt beliebig viele Argumente und fügt sie mit dem plattformspezifischen Trenner aneinander: `os.path.join("a", "b", "c")` ergibt unter Linux `"a/b/c"`, unter Windows `"a\\b\\c"`. Doppelte Trenner werden dabei vermieden — auch wenn du selbst einen einfügst.

> [!TIP]
> Gib niemals Pfade mit fest einprogrammierten Slashes weiter. `os.path.join` ist eine Zeile mehr, aber spart dir Stunden Debugging beim ersten Windows-Benutzer, der dein Skript ausführt.


In [8]:
import os

p1 = os.path.join("temp_messdaten", "messung_01.csv")
p2 = os.path.join("temp_messdaten", "archiv", "altlast.csv")
p3 = os.path.join("temp_messdaten", "logbuch.log")

print("p1:", p1)
print("p2:", p2)
print("p3:", p3, "existiert?", os.path.isfile(p3))


p1: temp_messdaten\messung_01.csv
p2: temp_messdaten\archiv\altlast.csv
p3: temp_messdaten\logbuch.log existiert? True


## 5. 🏁 Mini-Challenge: summiere alle Dateigrößen

Zum Abschluss eine kleine Übung, die alle bisherigen Bausteine kombiniert. Summiere die Dateigrößen aller Einträge in `temp_messdaten`, die **Dateien** (nicht Ordner!) sind, und weise das Ergebnis der Variable `summe_bytes` zu.

**Erwartet:** 164 Byte (= 35 + 28 + 42 + 29 + 30).


In [9]:
import os

basis = "temp_messdaten"

# TODO: iteriere ueber die Eintraege, summiere Dateigroessen
summe_bytes = 0

print("Gesamt:", summe_bytes, "Byte")

try:
    # erwartete Summe der oben angelegten Dateien: 35+28+42+29+30 = 164
    assert summe_bytes == 164, f"Erwartet 164 Byte, bekommen: {summe_bytes}"
    print("✅ Richtig!")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ `summe_bytes` fehlt noch.")


Gesamt: 0 Byte
❌ Noch nicht ganz: Erwartet 164 Byte, bekommen: 0


## 6. Aufräumen

Um dein Arbeitsverzeichnis sauber zu hinterlassen, entfernen wir den Temp-Ordner am Ende wieder. Das Modul **`shutil`** aus der Standardbibliothek enthält dafür die Funktion **`shutil.rmtree(pfad)`**, die einen Ordner *mit allen Unterordnern und Dateien* rekursiv löscht.


> [!WARNING]
> **`shutil.rmtree` fragt nicht nach!** Es löscht **unwiderruflich** alles unterhalb des angegebenen Pfades. Immer eine Sekunde nachdenken, bevor du den Aufruf ausführst, und ausschließlich auf Ordner anwenden, die du selbst angelegt hast.

Wir prüfen mit **`os.path.exists`** vorher, ob der Ordner überhaupt noch existiert — dann kannst du diese Zelle gefahrlos auch mehrfach ausführen.


In [10]:
import os
import shutil

ziel = "temp_messdaten"
if os.path.exists(ziel):
    shutil.rmtree(ziel)
    print(f"Ordner '{ziel}' wurde entfernt.")
else:
    print(f"Ordner '{ziel}' existiert nicht (mehr) — nichts zu tun.")


Ordner 'temp_messdaten' wurde entfernt.


## ✅ Zusammenfassung

- `os.listdir(pfad)` liefert den Ordnerinhalt als Liste von Namen (ohne Pfad-Präfix).
- `os.path.isfile(pfad)` und `os.path.isdir(pfad)` unterscheiden Dateien und Verzeichnisse.
- `os.path.getsize(pfad)` gibt die Dateigröße in Byte zurück.
- `os.path.join(basis, name)` setzt Pfade plattformunabhängig zusammen.
- `os.makedirs(pfad, exist_ok=True)` legt Ordner inklusive fehlender Zwischenebenen an.
- `shutil.rmtree(pfad)` löscht rekursiv — mit Bedacht einsetzen!

## ➡️ Nächster Schritt
Weiter mit [03_aufgaben.ipynb](03_aufgaben.ipynb) — fünf Übungsaufgaben rund um Dateien, Ordner und Ingenieurs-Datenformate.
